# Leitor de PDF com Gradio

Este notebook cria uma interface Gradio que recebe um PDF e devolve um áudio com a leitura do texto.

In [ ]:
# Instale as dependências necessárias
!pip -q install gradio pypdf gTTS

In [ ]:
import tempfile
from pathlib import Path

import gradio as gr
from gtts import gTTS
from pypdf import PdfReader


def pdf_para_audio(pdf_file, idioma):
    if pdf_file is None:
        raise gr.Error("Envie um arquivo PDF para continuar.")

    reader = PdfReader(pdf_file)
    texto = "".join(page.extract_text() or "" for page in reader.pages).strip()

    if not texto:
        raise gr.Error("Não foi possível extrair texto desse PDF.")

    tts = gTTS(text=texto, lang=idioma)
    temp_dir = Path(tempfile.mkdtemp())
    audio_path = temp_dir / "leitura.mp3"
    tts.save(str(audio_path))
    return str(audio_path)


demo = gr.Interface(
    fn=pdf_para_audio,
    inputs=[
        gr.File(label="PDF", file_types=[".pdf"]),
        gr.Dropdown(["pt", "en", "es"], value="pt", label="Idioma"),
    ],
    outputs=gr.Audio(label="Áudio gerado", type="filepath"),
    title="Leitor de PDF",
    description="Envie um PDF e receba o áudio com a leitura do texto.",
)

demo.launch()